## 嵌入向量

在我們之前的例子中，我們操作的是長度為 `vocab_size` 的高維度詞袋向量，並且我們明確地將低維度的位置表示向量轉換為稀疏的一熱編碼表示。這種一熱編碼表示並不節省記憶體，此外，每個詞語都是彼此獨立處理的，也就是說，一熱編碼向量不表達詞語之間的任何語義相似性。

在本單元中，我們將繼續探索 **News AG** 數據集。首先，讓我們加載數據，並從之前的筆記本獲取一些定義。


In [1]:
import torch
import torchtext
import numpy as np
from torchnlp import *
train_dataset, test_dataset, classes, vocab = load_dataset()
vocab_size = len(vocab)
print("Vocab size = ",vocab_size)

Loading dataset...


d:\WORK\ai-for-beginners\5-NLP\14-Embeddings\data\train.csv: 29.5MB [00:01, 18.8MB/s]                            
d:\WORK\ai-for-beginners\5-NLP\14-Embeddings\data\test.csv: 1.86MB [00:00, 11.2MB/s]                          


Building vocab...
Vocab size =  95812



## 甚麼是 embedding？

**Embedding** 的概念是用較低維度的密集向量來表示詞語，該向量某種程度上反映了詞語的語義意義。我們稍後會討論如何構建有意義的詞嵌入，但現階段先把 embedding 視為一種將詞向量降維的方法。

因此，embedding 層會以詞語作為輸入，並產生指定 `embedding_size` 的輸出向量。從某種意義上說，它與 `Linear` 層非常相似，但它不是接收 one-hot 編碼的向量，而是可以直接接收詞語編號作為輸入。

將 embedding 層作為神經網絡的第一層後，我們可以從散裝詞袋模式轉換到 **embedding bag** 模式。在這模式中，我們先將文本中每個詞語轉換成相應的 embedding，然後對這些 embedding 計算某種彙總函數，例如 `sum`、`average` 或 `max`。

![顯示五個序列詞語的 embedding 分類器的圖片](../../../../../translated_images/zh-HK/embedding-classifier-example.b77f021a7ee67eee.webp)

我們的分類器神經網絡將從 embedding 層開始，接著是彙總層，頂層則為線性分類器：


In [2]:
class EmbedClassifier(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, num_class):
        super().__init__()
        self.embedding = torch.nn.Embedding(vocab_size, embed_dim)
        self.fc = torch.nn.Linear(embed_dim, num_class)

    def forward(self, x):
        x = self.embedding(x)
        x = torch.mean(x,dim=1)
        return self.fc(x)

### 處理可變序列長度

因為這個架構，我們必須用特定的方式來建立送入網絡的迷你批次。在上一單元中，使用詞袋模型時，迷你批次中的所有詞袋張量大小均相等 `vocab_size`，不論我們的文本序列實際長度為何。當我們改用詞嵌入時，每個文本範例的詞數將會不一樣，而當要將這些範例合併成迷你批次時，就必須施加一些填充。

這可以用提供 `collate_fn` 函數給資料來源的相同技巧來完成：


In [3]:
def padify(b):
    # b is the list of tuples of length batch_size
    #   - first element of a tuple = label, 
    #   - second = feature (text sequence)
    # build vectorized sequence
    v = [encode(x[1]) for x in b]
    # first, compute max length of a sequence in this minibatch
    l = max(map(len,v))
    return ( # tuple of two tensors - labels and features
        torch.LongTensor([t[0]-1 for t in b]),
        torch.stack([torch.nn.functional.pad(torch.tensor(t),(0,l-len(t)),mode='constant',value=0) for t in v])
    )

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=padify, shuffle=True)

### 訓練嵌入分類器

現在我們已經定義了合適的數據加載器，可以使用上一單元中定義的訓練函數來訓練模型：


In [4]:
net = EmbedClassifier(vocab_size,32,len(classes)).to(device)
train_epoch(net,train_loader, lr=1, epoch_size=25000)

3200: acc=0.6415625
6400: acc=0.6865625
9600: acc=0.7103125
12800: acc=0.726953125
16000: acc=0.739375
19200: acc=0.75046875
22400: acc=0.7572321428571429


(0.889799795315499, 0.7623160588611644)

> <strong>注意</strong>：我們這裡只訓練 25k 筆記錄（不到一個完整的 epoch）以節省時間，但你可以繼續訓練，編寫一個函數來訓練多個 epoch，並嘗試調整學習率參數以達到更高的準確率。你應該能夠達到大約 90% 的準確率。


### EmbeddingBag 層與可變長序列表示

在之前的架構中，我們需要將所有序列填充至相同長度，以便放入小批次中。這不是表示可變長序列的最有效方法——另一種方法是使用 **offset** 向量，該向量會保存存儲於一個大型向量中的所有序列的偏移量。

![展示 offset 序列表示的圖片](../../../../../translated_images/zh-HK/offset-sequence-representation.eb73fcefb29b46ee.webp)

> <strong>注意</strong>：上圖中顯示的是一個字元序列，但在我們的範例中，我們處理的是詞序列。然而，使用 offset 向量表示序列的一般原理保持不變。

為了使用 offset 表示，我們使用 [`EmbeddingBag`](https://pytorch.org/docs/stable/generated/torch.nn.EmbeddingBag.html) 層。它類似於 `Embedding`，但它接受內容向量和 offset 向量作為輸入，且它還包含一個平均層，該層可以是 `mean`、`sum` 或 `max`。

以下是使用 `EmbeddingBag` 的修改後網絡：


In [5]:
class EmbedClassifier(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, num_class):
        super().__init__()
        self.embedding = torch.nn.EmbeddingBag(vocab_size, embed_dim)
        self.fc = torch.nn.Linear(embed_dim, num_class)

    def forward(self, text, off):
        x = self.embedding(text, off)
        return self.fc(x)

為了準備訓練所需的數據集，我們需要提供一個轉換函數來準備偏移向量：


In [6]:
def offsetify(b):
    # first, compute data tensor from all sequences
    x = [torch.tensor(encode(t[1])) for t in b]
    # now, compute the offsets by accumulating the tensor of sequence lengths
    o = [0] + [len(t) for t in x]
    o = torch.tensor(o[:-1]).cumsum(dim=0)
    return ( 
        torch.LongTensor([t[0]-1 for t in b]), # labels
        torch.cat(x), # text 
        o
    )

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=offsetify, shuffle=True)

注意，與之前所有範例不同，我們的網絡現在接受兩個參數：數據向量和偏移向量，它們大小不同。同樣，我們的數據加載器也提供我們3個數值而非2個：文本向量和偏移向量都作為特徵提供。因此，我們需要稍微調整訓練函數來處理這一點：


In [7]:
net = EmbedClassifier(vocab_size,32,len(classes)).to(device)

def train_epoch_emb(net,dataloader,lr=0.01,optimizer=None,loss_fn = torch.nn.CrossEntropyLoss(),epoch_size=None, report_freq=200):
    optimizer = optimizer or torch.optim.Adam(net.parameters(),lr=lr)
    loss_fn = loss_fn.to(device)
    net.train()
    total_loss,acc,count,i = 0,0,0,0
    for labels,text,off in dataloader:
        optimizer.zero_grad()
        labels,text,off = labels.to(device), text.to(device), off.to(device)
        out = net(text, off)
        loss = loss_fn(out,labels) #cross_entropy(out,labels)
        loss.backward()
        optimizer.step()
        total_loss+=loss
        _,predicted = torch.max(out,1)
        acc+=(predicted==labels).sum()
        count+=len(labels)
        i+=1
        if i%report_freq==0:
            print(f"{count}: acc={acc.item()/count}")
        if epoch_size and count>epoch_size:
            break
    return total_loss.item()/count, acc.item()/count


train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6153125
6400: acc=0.6615625
9600: acc=0.6932291666666667
12800: acc=0.715078125
16000: acc=0.7270625
19200: acc=0.7382291666666667
22400: acc=0.7486160714285715


(22.771553103007037, 0.7551983365323096)


## 語義嵌入：Word2Vec

在之前的例子中，模型的嵌入層學會將單詞映射到向量表示，但這種表示並沒有多少語義意義。最好能學習出這種向量表示，使得相似的單詞或同義詞在某種向量距離（例如歐氏距離）上彼此接近。

為此，我們需要以特定方式在大量文本上預訓練我們的嵌入模型。最早用來訓練語義嵌入的方法之一叫做 [Word2Vec](https://en.wikipedia.org/wiki/Word2vec)。這基於兩種主要架構，用來產生分散式的詞向量表示：

 - <strong>連續詞袋模型</strong> (CBoW) — 在這種架構中，我們訓練模型從周圍上下文中預測一個詞。給定 ngram $(W_{-2},W_{-1},W_0,W_1,W_2)$，模型的目標是從 $(W_{-2},W_{-1},W_1,W_2)$ 預測 $W_0$。
 - <strong>連續跳字模型</strong> 與 CBoW 相反。模型利用周圍上下文窗口的詞來預測當前詞。

CBoW 較快，而跳字模型較慢，但在表示不常見詞彙方面效果更好。

![顯示 CBoW 與跳字演算法將詞轉換為向量的圖片。](../../../../../translated_images/zh-HK/example-algorithms-for-converting-words-to-vectors.fbe9207a726922f6.webp)

若要實驗預先在 Google News 資料集上訓練的 word2vec 嵌入，我們可以使用 **gensim** 庫。下面找出與 'neural' 最相似的單詞

> **注意：** 當你首次建立詞向量時，下載可能會花一些時間！


In [8]:
import gensim.downloader as api
w2v = api.load('word2vec-google-news-300')

In [9]:
for w,p in w2v.most_similar('neural'):
    print(f"{w} -> {p}")

neuronal -> 0.7804799675941467
neurons -> 0.7326500415802002
neural_circuits -> 0.7252851724624634
neuron -> 0.7174385190010071
cortical -> 0.6941086649894714
brain_circuitry -> 0.6923246383666992
synaptic -> 0.6699118614196777
neural_circuitry -> 0.6638563275337219
neurochemical -> 0.6555314064025879
neuronal_activity -> 0.6531826257705688


我們亦可以從詞語計算向量嵌入，以用於訓練分類模型（為清晰起見，我們只顯示向量的前20個分量）：


In [10]:
w2v.word_vec('play')[:20]

array([ 0.01226807,  0.06225586,  0.10693359,  0.05810547,  0.23828125,
        0.03686523,  0.05151367, -0.20703125,  0.01989746,  0.10058594,
       -0.03759766, -0.1015625 , -0.15820312, -0.08105469, -0.0390625 ,
       -0.05053711,  0.16015625,  0.2578125 ,  0.10058594, -0.25976562],
      dtype=float32)

语义嵌入的优点是你可以通过操作向量编码来改变语义。例如，我们可以要求找到一个词，其向量表示尽可能接近词 *king* 和 *woman*，并且尽可能远离词 *man*：


In [10]:
w2v.most_similar(positive=['king','woman'],negative=['man'])[0]

('queen', 0.7118192911148071)

CBoW 和 Skip-Grams 都是「預測式」嵌入，因為它們只考慮局部上下文。Word2Vec 並沒有利用全局上下文。

**FastText** 建立在 Word2Vec 之上，透過學習每個詞以及詞內的字元 n-gram 的向量表示。在每一次訓練步驟中，這些表示的值會被平均成一個向量。雖然這為預訓練增加了大量額外計算，但它使詞嵌入能夠編碼子詞資訊。

另一個方法，**GloVe**，利用共現矩陣的概念，使用神經方法將共現矩陣分解成更具表現力且非線性的詞向量。

你可以通過將嵌入改成 FastText 和 GloVe 來玩玩這個範例，因為 gensim 支援多種不同的詞嵌入模型。


## 在 PyTorch 中使用預先訓練的嵌入

我們可以修改上面範例，使用語意嵌入（例如 Word2Vec）來預先填充嵌入層的矩陣。需要考慮的是，預先訓練嵌入的詞彙和我們的文本語料庫的詞彙很可能不匹配，因此我們會使用隨機值來初始化缺失詞彙的權重：


In [11]:
embed_size = len(w2v.get_vector('hello'))
print(f'Embedding size: {embed_size}')

net = EmbedClassifier(vocab_size,embed_size,len(classes))

print('Populating matrix, this will take some time...',end='')
found, not_found = 0,0
for i,w in enumerate(vocab.get_itos()):
    try:
        net.embedding.weight[i].data = torch.tensor(w2v.get_vector(w))
        found+=1
    except:
        net.embedding.weight[i].data = torch.normal(0.0,1.0,(embed_size,))
        not_found+=1

print(f"Done, found {found} words, {not_found} words missing")
net = net.to(device)

Embedding size: 300
Populating matrix, this will take some time...Done, found 41080 words, 54732 words missing


現在讓我們訓練模型。請注意，訓練模型所需的時間比之前的範例明顯更長，這是因為嵌入層尺寸較大，因此參數數量也大幅增加。此外，也因此，如果我們想避免過度擬合，可能需要使用更多的範例來訓練模型。


In [12]:
train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6359375
6400: acc=0.68109375
9600: acc=0.7067708333333333
12800: acc=0.723671875
16000: acc=0.73625
19200: acc=0.7463541666666667
22400: acc=0.7560714285714286


(214.1013875559821, 0.7626759436980166)

在我們的情況中，我們沒有看到準確度有大幅提升，這很可能是因為詞彙表相差甚遠。
為了克服詞彙表不同的問題，我們可以採用以下其中一種解決方案：
* 在我們的詞彙表上重新訓練 word2vec 模型
* 使用預先訓練好的 word2vec 模型的詞彙表來載入我們的數據集。載入數據集時可以指定要使用的詞彙表。

後一種方法似乎較為簡單，尤其是因為 PyTorch `torchtext` 框架內建對 embeddings 的支援。例如，我們可以用以下方式實例化基於 GloVe 的詞彙表：


In [14]:
vocab = torchtext.vocab.GloVe(name='6B', dim=50)

100%|█████████▉| 399999/400000 [00:15<00:00, 25411.14it/s]


載入的詞彙庫具有以下基本操作：
* `vocab.stoi` 允許我們將字詞轉換為其詞典索引
* `vocab.itos` 則相反 — 將數字轉換為字詞
* `vocab.vectors` 是嵌入向量的陣列，所以要取得字詞 `s` 的嵌入，我們需要使用 `vocab.vectors[vocab.stoi[s]]`

下面是操作嵌入向量的範例，用以證明方程式 **kind-man+woman = queen**（我必須稍微調整係數才能使其成立）：


In [15]:
# get the vector corresponding to kind-man+woman
qvec = vocab.vectors[vocab.stoi['king']]-vocab.vectors[vocab.stoi['man']]+1.3*vocab.vectors[vocab.stoi['woman']]
# find the index of the closest embedding vector 
d = torch.sum((vocab.vectors-qvec)**2,dim=1)
min_idx = torch.argmin(d)
# find the corresponding word
vocab.itos[min_idx]

'queen'

要使用這些嵌入來訓練分類器，我們首先需要使用 GloVe 詞彙編碼我們的數據集： 


In [16]:
def offsetify(b):
    # first, compute data tensor from all sequences
    x = [torch.tensor(encode(t[1],voc=vocab)) for t in b] # pass the instance of vocab to encode function!
    # now, compute the offsets by accumulating the tensor of sequence lengths
    o = [0] + [len(t) for t in x]
    o = torch.tensor(o[:-1]).cumsum(dim=0)
    return ( 
        torch.LongTensor([t[0]-1 for t in b]), # labels
        torch.cat(x), # text 
        o
    )

正如我們上面所見，所有向量嵌入都儲存在 `vocab.vectors` 矩陣中。這使得使用簡單複製將這些權重加載到嵌入層的權重中變得非常容易：


In [17]:
net = EmbedClassifier(len(vocab),len(vocab.vectors[0]),len(classes))
net.embedding.weight.data = vocab.vectors
net = net.to(device)

而家我哋訓練我哋嘅模型，睇下有冇得到更好嘅結果：


In [18]:
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=offsetify, shuffle=True)
train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6271875
6400: acc=0.68078125
9600: acc=0.7030208333333333
12800: acc=0.71984375
16000: acc=0.7346875
19200: acc=0.7455729166666667
22400: acc=0.7529464285714286


(35.53972978646833, 0.7575175943698017)

我們未見準確度顯著提升的原因之一，是因為我們資料集中有些詞彙並不在預先訓練好的 GloVe 詞彙表中，因此基本上被忽略了。為了克服這個問題，我們可以在我們的資料集上自行訓練嵌入向量。 


## 上下文嵌入

傳統預訓練嵌入表示方法如 Word2Vec 的一個主要限制是字義消歧問題。雖然預訓練嵌入可以捕捉到字詞在上下文中的部分意義，但一個字詞的所有可能含義都被編碼到同一個嵌入中。這可能在下游模型中造成問題，因為許多字詞例如「play」根據使用的上下文有不同的意思。

例如字詞「play」在以下兩個不同句子中的意思就大不相同：
- 我去了劇院看一場<strong>戲劇</strong>。
- 約翰想要和他的朋友們<strong>玩耍</strong>。

上述預訓練嵌入對「play」這個詞的兩種意思都表示為相同的嵌入。為克服此限制，我們需要基於<strong>語言模型</strong>來建構嵌入，該模型是在大量文本語料上訓練，並且<em>了解</em>詞語在不同上下文中如何組合。本教程不會深入討論上下文嵌入，但我們會在下一單元談論語言模型時再次提及。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
本文件由 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 翻譯而成。雖然我們致力於確保準確性，但請注意，機器自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應被視為權威來源。對於重要資訊，建議進行專業人工翻譯。我們不對因使用本翻譯而產生的任何誤解或誤釋承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
